# 03 · Entender la cohorte completa antes de entrenar redes

Estudiamos todas las imágenes de las doce prendas que han superado los controles. Sus tamaños de clase son diferentes: **no las igualamos mediante un cupo**. Exploramos fotos de train y medimos baselines en validation. No cargamos fotografías ni predicciones de test.

La cohorte corresponde al repertorio elegido y a sus controles de calidad; no es todo el catálogo ni una medida de demanda comercial.

> **Ejecución:** pendiente de volver a ejecutar con la cohorte completa y el protocolo coherente. Los resultados anteriores no se usan como evidencia de esta cohorte.

## Papel de este EDA en las vías del proyecto

Según nuestra interpretación de las pistas recogidas en clase, la prioridad es entender el problema y disponer de referencias que podamos explicar, no maximizar una puntuación. Los baselines clásicos permiten medir qué señal existe antes de aprender representaciones. No exigimos que todas las redes los superen: un resultado peor también informa sobre optimización, representación o tamaño muestral. El mapa de scratch, preentrenamiento y extensiones está en 01. Aquí se trabajará con train para explorar y validation para comparar, sin inspeccionar el test para tomar decisiones.


## 1. Protocolo, publicación y entorno

Macro-F1 es la métrica primaria. Accuracy, resultados por clase y coste completan la interpretación. El azar uniforme (1/12) es una expectativa teórica, no nuestro único baseline. El dummy aprende la clase mayoritaria de train; el segundo baseline será regresión logística sobre histogramas de color, que no modelan la disposición espacial. No se ajustan sus hiperparámetros buscando el mejor resultado en validation.

In [ ]:
from pathlib import Path
import sys,json,time,hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display,Markdown
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib','inline')
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/data/local_dataset.py').is_file())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'src/data'))
from notebook_utils import setup,styled,save_figure,es,BLUE,ORANGE
from data_contract import sha256
setup()
protocol=json.loads((ROOT/'config/experimental_protocol_v1.json').read_text())
output=ROOT/'data/processed'
pointer=json.loads((output/'CURRENT.json').read_text())
run=output/pointer['run_dir']
checks=json.loads((run/'checksums.json').read_text())
assert sha256(run/'image_manifest.csv.gz')==checks['image_manifest.csv.gz']
manifest=pd.read_csv(run/'image_manifest.csv.gz',dtype={'article_id':'string','product_code':'string','group_id':'string'},keep_default_na=False)
train=manifest.loc[manifest.split.eq('train')].copy()
validation=manifest.loc[manifest.split.eq('validation')].copy()
labels=json.loads((run/'labels.json').read_text())
names=sorted(labels,key=labels.get)
RESULTS=ROOT/'results/metrics/03_eda';RESULTS.mkdir(parents=True,exist_ok=True)
SEED=42
assert set(train.group_id).isdisjoint(validation.group_id)
display(Markdown(f"**Publicación:** `{run.name}` · Train: {es(len(train))} · Validation: {es(len(validation))}."))
display(pd.Series(protocol['default_loss'],name='Pérdida inicial de las futuras redes').to_frame())

## 2. Cuántas fotos conserva cada clase

La tabla sigue la selección: fotos válidas del tipo elegido, fotos que conservamos tras los controles y porcentaje de cada clase en la cohorte. Las diferencias responden a incidencias y duplicados; no a un objetivo de igualar frecuencias. El número final sale de las reglas.

Las frecuencias describen las prendas seleccionadas después de los controles. Validation y test mantienen grupos completos y proporciones de clase aproximadas; no reponderamos predicciones de una muestra equilibrada para simular otra población.

In [ ]:
selected=manifest.loc[manifest.split.ne('excluded')]
valid_scope=manifest.loc[manifest.image_valid & manifest.label.isin(names)]
counts=pd.DataFrame({'Válidas en origen':valid_scope.label.value_counts(),'Conservadas':selected.label.value_counts()}).loc[names]
counts['Conservadas (%)']=100*counts.Conservadas/counts.Conservadas.sum()
counts['Retenidas de las válidas (%)']=100*counts.Conservadas/counts['Válidas en origen']
display(styled(counts,2))
counts.to_csv(RESULTS/'sampling_distribution.csv')
priors=(counts.Conservadas/counts.Conservadas.sum()).to_dict()
(RESULTS/'eligible_population_priors.json').write_text(json.dumps({'population':'all selected garment types after common quality checks, no caps','priors':priors,'status':'descriptive cohort frequencies, not demand or entire catalog'},indent=2))
weights=json.loads((run/'training_class_weights.json').read_text())
display(Markdown(f"Pesos opcionales de la pérdida: **{es(min(weights.values()),3)} a {es(max(weights.values()),3)}**. Con frecuencias desiguales pueden alejarse de uno; la referencia sigue sin pesos y H4 medirá su efecto."))

## 3. Revisión visual ampliada y sistemática de train

Se fijan ocho grupos independientes por clase y un representante por grupo: 96 fotografías, frente a los dos ejemplos iniciales por clase. La muestra se elige con semilla antes de examinar errores de modelos. Se conserva la misma pauta para todas las clases, incluidas Shirt/T-shirt y Sweater/Cardigan.

La galería permite una revisión humana de contorno, textura, fondo, encuadre y orientación. No proporciona etiquetas verificadas de pose o edad ni una tasa global de ruido. Se genera una plantilla de anotación sin sobrescribir anotaciones previas. Marcar «incierto» cuando proceda; antes de concluir que una clase es incoherente, revisar sistemáticamente estas anotaciones y ampliar la muestra si no es suficiente. La revisión específica de Top sigue pendiente y debe usar productos reservados fuera del modelado.

In [ ]:
review=[]
for label,part in train.groupby('label',sort=True):
    representatives=part.sort_values('article_id').drop_duplicates('group_id')
    review.append(representatives.sample(n=8,random_state=SEED))
review=pd.concat(review).sort_values(['label','article_id'])
assert len(review)==96 and review.group_id.nunique()==96
review_path=RESULTS/'training_visual_review_96.csv'
columns=['article_id','product_code','group_id','label','image_path']
if not review_path.exists():
    annotation=review[columns].copy()
    for field in ['background_type','pose_or_presentation','label_consistency','uncertainty','notes']:
        annotation[field]='pending'
    annotation.to_csv(review_path,index=False)
else:
    existing=pd.read_csv(review_path,dtype={'article_id':'string'})
    assert set(existing.article_id)==set(review.article_id),'No sobrescribir revisión de otra muestra'
for page in range(4):
    part=review.iloc[page*24:(page+1)*24]
    fig,axes=plt.subplots(3,8,figsize=(22,12))
    for ax,row in zip(axes.flat,part.itertuples()):
        with Image.open(output/row.image_path) as image:ax.imshow(image)
        ax.set_title(f'{row.label}\n{row.article_id}')
        ax.axis('off')
    fig.suptitle(f'Revisión de train: ocho productos por clase · Bloque {page+1}')
    save_figure(fig,ROOT,f'03_train_review_{page+1}','Fuente: muestra fija de train H&M · La revisión humana se registra aparte; no se muestran fotos de test')
    plt.show()
display(Markdown(f"**Plantilla:** `{review_path.relative_to(ROOT)}`. Las anotaciones pendientes no se interpretan como observaciones confirmadas."))

## 4. Segmentos comerciales: prendas infantiles y adultas

`index_group_name` es una categoría comercial suministrada por H&M, no una edad inferida de la fotografía. Su distribución por clase permite detectar si la comparación mezcla líneas de producto muy diferentes. No atribuimos una pose, edad o calidad visual a una imagen a partir de esa columna. Estas diferencias pueden actuar como factores de confusión y requieren interpretación junto a la galería.

In [ ]:
segments=pd.crosstab(train.label,train.index_group_name).reindex(names)
display(styled(segments))
display(styled(100*segments.div(segments.sum(axis=1),axis=0),1))
segments.to_csv(RESULTS/'training_commercial_segments.csv')

## 5. Características simples y variación dentro de cada clase

Se extraen histogramas RGB de 16 intervalos por canal (48 características). Son un baseline de color, no embeddings profundos. También se registran brillo, variabilidad de intensidad y brillo de las esquinas del contenido aproximado, retirando el relleno añadido según la relación de aspecto original. Esa última variable es un **proxy**, no una segmentación verificada del fondo: prendas claras y píxeles de la prenda pueden ocupar las esquinas.

Los descriptores de train permiten examinar variabilidad; no bastan para concluir qué clases confundirá una CNN. Las poses no se cuantifican automáticamente sin anotaciones. La extracción puede tardar la primera vez; se reutiliza una caché ligada al manifiesto y a esta definición de características.

In [ ]:
feature_version='rgbhist16-content-proxies-v1'
fingerprint=hashlib.sha256((sha256(run/'image_manifest.csv.gz')+feature_version).encode()).hexdigest()[:16]
cache=RESULTS/f'features_{fingerprint}.npz'
rows=pd.concat([train,validation],ignore_index=True)
start=time.perf_counter()
if cache.exists():
    data=np.load(cache)
    assert data['article_ids'].tolist()==rows.article_id.tolist()
    X=data['X'];descriptors=data['descriptors']
else:
    features=[];stats=[]
    for i,row in enumerate(rows.itertuples(),1):
        with Image.open(output/row.image_path) as source:
            image=source.convert('RGB');size=image.width
            scale=size/max(row.width,row.height)
            w=max(1,round(row.width*scale));h=max(1,round(row.height*scale))
            left=(size-w)//2;top=(size-h)//2
            content=np.asarray(image.crop((left,top,left+w,top+h)).resize((32,32)),dtype=np.float32)/255
        features.append(np.concatenate([np.histogram(content[:,:,c],bins=16,range=(0,1))[0]/1024 for c in range(3)]))
        corners=np.concatenate([content[:4,:4].ravel(),content[:4,-4:].ravel(),content[-4:,:4].ravel(),content[-4:,-4:].ravel()])
        stats.append([content.mean(),content.std(),corners.mean()])
    X=np.asarray(features,dtype=np.float32);descriptors=np.asarray(stats,dtype=np.float32)
    np.savez_compressed(cache,X=X,descriptors=descriptors,article_ids=rows.article_id.to_numpy(dtype=str))
train_X=X[:len(train)];val_X=X[len(train):]
train_y=train.label_id.astype(int).to_numpy();val_y=validation.label_id.astype(int).to_numpy()
profile=pd.DataFrame(descriptors[:len(train)],columns=['brillo','variacion_intensidad','esquinas_contenido'])
profile['label']=train.label.to_numpy()
display(styled(profile.groupby('label').agg(['median','std']),3))
print('Extracción/carga completada. Forma de características:',X.shape)

## 6. Baselines fijados: mayoría y regresión logística de color

El escalado se ajusta solo con train. La regresión logística usa C=1 y un máximo de 500 iteraciones, sin pesos de clase ni búsqueda de hiperparámetros. Si no converge se informa: no se presenta silenciosamente como un ajuste satisfactorio. La medición de tiempo excluye extracción de características y no representa el coste total de una red. Estos baselines no tienen entrenamiento GPU.

Los intervalos de validation remuestrean grupos de producto completos, manteniendo juntas sus variantes. Son intervalos del conjunto y ajuste actuales, no variabilidad entre entrenamientos ni una prueba de superioridad. Validation sigue disponible para desarrollo; estos números no son resultados finales de test.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from threadpoolctl import threadpool_limits
models={'Mayoría train':DummyClassifier(strategy='most_frequent'),
        'Logística de color':make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=500,class_weight=None,random_state=SEED))}
results=[];predictions={};per_class={}
def group_intervals(y,pred,groups,draws=300):
    unique,inverse=np.unique(groups,return_inverse=True)
    cubes=np.zeros((len(unique),len(names),len(names)),dtype=int)
    np.add.at(cubes,(inverse,y,pred),1)
    rng=np.random.default_rng(SEED);samples=[]
    for _ in range(draws):
        cm=cubes[rng.integers(len(unique),size=len(unique))].sum(axis=0)
        tp=cm.diagonal();den=cm.sum(0)+cm.sum(1)
        samples.append([tp.sum()/cm.sum(),np.divide(2*tp,den,out=np.zeros(len(names),dtype=float),where=den>0).mean()])
    return np.quantile(samples,[.025,.975],axis=0)
with threadpool_limits(limits=2):
    for name,model in models.items():
        start=time.perf_counter();model.fit(train_X,train_y);fit_seconds=time.perf_counter()-start
        start=time.perf_counter();pred=model.predict(val_X);predict_seconds=time.perf_counter()-start
        predictions[name]=pred
        ci=group_intervals(val_y,pred,validation.group_id.to_numpy())
        results.append({'Modelo':name,'accuracy':accuracy_score(val_y,pred),'macro_F1':f1_score(val_y,pred,labels=np.arange(len(names)),average='macro',zero_division=0),
                        'accuracy_IC95_inf':ci[0,0],'accuracy_IC95_sup':ci[1,0],'F1_IC95_inf':ci[0,1],'F1_IC95_sup':ci[1,1],
                        'ajuste_s':fit_seconds,'prediccion_ms_imagen':1000*predict_seconds/len(val_y)})
        per_class[name]=classification_report(val_y,pred,labels=np.arange(len(names)),target_names=names,zero_division=0,output_dict=True)
metrics=pd.DataFrame(results).set_index('Modelo');display(styled(metrics,4))
metrics.to_csv(RESULTS/'validation_baselines.csv')
report=pd.DataFrame(per_class['Logística de color']).T;display(styled(report.loc[names],3))
report.to_csv(RESULTS/'validation_colour_per_class.csv')
logistic=models['Logística de color'].named_steps['logisticregression']
converged=bool(logistic.n_iter_.max()<logistic.max_iter)
display(Markdown(f"**Azar uniforme esperado:** {es(100/len(names),2)} % de accuracy; referencia teórica. **Logística dentro del límite de iteraciones:** {converged}."))
np.savez_compressed(RESULTS/'validation_predictions.npz',article_ids=validation.article_id.to_numpy(dtype=str),y=val_y,**{f'model_{i}':pred for i,pred in enumerate(predictions.values())})

## 7. Confusión de un baseline: evidencia limitada a su representación

Una confusión de histogramas de color no demuestra una ambigüedad intrínseca entre prendas: el baseline ignora su disposición espacial. Se publican recuentos y proporciones por clase, para distinguir un fallo sistemático de un efecto del soporte. Las parejas Shirt/T-shirt y Sweater/Cardigan son hipótesis a contrastar, no las seleccionamos a posteriori como si se hubieran predicho a partir de estos resultados.

In [ ]:
cm=confusion_matrix(val_y,predictions['Logística de color'],labels=np.arange(len(names)))
cm_normalized=cm/cm.sum(axis=1,keepdims=True)
fig,axes=plt.subplots(1,2,figsize=(24,11))
for ax,values,title in zip(axes,[cm,cm_normalized],['Recuentos','Proporción por clase real']):
    image=ax.imshow(values,cmap='Blues',vmin=0,vmax=None if title=='Recuentos' else 1)
    ax.set_xticks(range(len(names)),names,rotation=70,ha='right');ax.set_yticks(range(len(names)),names)
    ax.set(xlabel='Clase predicha',ylabel='Clase real',title=title)
    ax.grid(False)
    fig.colorbar(image,ax=ax,shrink=.7)
fig.suptitle('La referencia de color no utiliza la estructura espacial')
save_figure(fig,ROOT,'03_colour_confusion','Fuente: validation H&M · Baseline fijo de histogramas RGB · No es una evaluación de test')
plt.show()
pairs=pd.DataFrame([{'Real':names[i],'Predicha':names[j],'Errores':int(cm[i,j]),'Fraccion_clase':float(cm_normalized[i,j])} for i in range(len(names)) for j in range(len(names)) if i!=j]).sort_values('Errores',ascending=False)
display(styled(pairs.head(12),3))

## 8. Qué sabemos y qué falta

La muestra visual de 96 grupos de train, la composición comercial y los descriptores ayudan a estudiar diferencias entre prendas. Las anotaciones humanas pendientes no son una revisión terminada. Los dos baselines se miden en validation con intervalos que remuestrean productos completos.

La distribución desigual de clases forma parte del problema: por eso leemos macro-F1, accuracy y resultados por clase juntos. Los pesos o el balance del sampler son hipótesis de entrenamiento, no excusas para eliminar fotos válidas.

El siguiente notebook mide tiempo y memoria de una época piloto antes de decidir si el coste exige una muestra de train. Si cabe entrenar con toda la cohorte, se conserva completa. El test predictivo permanece cerrado.